# Genomic MIL: validación consciente del linaje

Protocolo prospectivamente especificado para resistencia binaria (`0 = susceptible`, `1 = resistente`). Comparación de cinco folds aleatorios estratificados y cinco folds bloqueados por ST **sobre la misma cohorte explícita**. No se reconstruye la cohorte desde predicciones históricas. El modo predeterminado es **preflight CPU, sin descargar modelos ni entrenar**.

La arquitectura NTv3 MultiHead MIL conserva CLS de 768 dimensiones, LayerNorm, dropout 0.15, cuatro cabezas `768 → 128 → 1` (Tanh, dropout 0.1), concatenación, proyección `3072 → 768`, LayerNorm y clasificador de dos clases. Se usa focal loss (α=.55, γ=2) y suma de similitudes coseno entre cabezas (λ=.01). `species_ids=6` se pasa como vector por secuencia para K. pneumoniae; confirmar especie antes de usar otro organismo.

**Unidad experimental:** genoma; RGI proporciona múltiples instancias por genoma. Selección de bolsas independiente de etiquetas, umbral fijo .5, ninguna selección de época/parámetros con outer test. XGB usa embeddings del MIL seleccionado dentro del fold: no es un baseline de foundation independiente. Los folds no se consideran réplicas independientes.

In [ ]:
# Instalación convencional Colab por defecto, sin scripts remotos ni pins NumPy.
INSTALL_DEPENDENCIES = globals().get('INSTALL_DEPENDENCIES', True)
INSTALL_SHAP = globals().get('INSTALL_SHAP', False)  # CLI installs are explicit and separate.
if INSTALL_DEPENDENCIES:
    import subprocess, sys, importlib.metadata
    loaded_before = {name:getattr(sys.modules.get(name), '__version__', None)
                     for name in ['numpy','pandas','sklearn','transformers','xgboost','shap']}
    packages = ['pandas>=2.2,<3', 'scikit-learn==1.6.1', 'transformers==4.56.2',
                'xgboost==3.0.5', 'openpyxl>=3.1,<4']
    if INSTALL_SHAP: packages.append('shap==0.48.0')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *packages])
    distributions = {'sklearn':'scikit-learn'}
    changed_loaded = [name for name,version in loaded_before.items() if version is not None
                      and importlib.metadata.version(distributions.get(name,name)) != version]
    assert not changed_loaded, (
        f'Paquetes ya importados cambiaron: {changed_loaded}. Reiniciar runtime y Run all antes de continuar.')
# No se fuerza downgrade de NumPy ni reinstalación de PyTorch del runtime.
# Si pip informa incompatibilidades, resolverlas/reiniciar antes de ejecutar el protocolo.


## 1. Rutas reales y preparación automática

Los defaults provienen del flujo RGI existente. Base: `/content/drive/MyDrive/rgi_Klebsiella_pneumoniae_ertapenem`. Se leen los originales `/content/drive/MyDrive/rgi_Kp_ertapenem_Susceptible` (label=0) y `/content/drive/MyDrive/rgi_Kp_ertapenem_Resistente` (label=1): la **procedencia explícita S/R es la fuente del fenotipo**, no una inferencia desde genes. Drive se monta por defecto en Colab. Se instalan paquetes convencionales por defecto, sin forzar pins de NumPy ni reinstalar PyTorch; si paquetes ya importados cambiaron, se exige reiniciar runtime.

No se necesitan `manifest.csv` ni `rgi_sequences.csv` preexistentes: se generan bajo `validated_results`, en `<run_id>/inputs` (CLI preflight/smoke usan subcarpetas separadas), sin modificar `dataset_limpio`. Se descubre cada `.txt/.tsv` del nivel superior de ambas carpetas, incluidos archivos pequeños/vacíos; cada uno tiene una fila en el inventario completo y, por defecto, en el manifiesto explícito (`genome_id,label,ST` y procedencia). No se usan automáticamente OOF, antiguos train/val ni cohortes filtradas para limitar aislados. Solo `cohort_selection` explícito puede restringir por nombres originales exactos con puente de procedencia auditado.

RGI TSV: columnas exactas `Cut_Off`, `Predicted_DNA`, opcional `Drug Class`. Defaults históricos: tamaño **>500 bytes**, hits `Perfect/Strict`, DNA **>20 bp**; se conserva cada archivo excluido y el motivo, y cada fila no elegible se audita. DNA nucleotídico IUPAC en mayúsculas; errores de parsing/esquema/DNA abortan, no se omiten silenciosamente. No se elimina DNA compartido entre S/R. Distintas filas/loci con la misma secuencia siguen siendo instancias distintas, con archivo y ordinal de registro TSV (cabecera=1); no se afirma que ese ordinal sea una línea física si hay campos multilinea.

Si los originales faltan o no tienen archivos, solo `allow_curated_fallback=True` permite usar **ambas** carpetas `dataset_limpio/Susceptible` y `dataset_limpio/Resistente`; hay warning y marca de fallback. No se mezclan fuentes. Esas copias pueden haber omitido vacíos: este modo no puede reclamar cobertura completa de los originales.

ST se lee de `dataset_limpio/ST_table.csv` y/o `.xlsx`, esquema real generado `accession,ST`. Si existen ambos, se verifica igualdad del mapa canónico; conflictos o duplicados abortan. No se ejecuta MLST ni se inventan linajes. IDs: solo un accession versionado distinto `GCA_/GCF_` inequívoco en el nombre (repeticiones del mismo literal no son aliases nuevos); de otro modo, quitar únicamente `.txt/.tsv` terminal y `_rgi` terminal. Varios accessions, colisiones entre archivos, conflictos S/R y mapeos ST ambiguos abortan. No se colapsan GCA con GCF, versiones ni se infieren equivalencias sin evidencia. Cruce manifiesto↔ST **one-to-one**; manifiesto↔bolsas one-to-one y secuencias↔manifiesto many-to-one validado.

ST no encontrado, vacío, `-`, `?`, `ST-` o desconocido se conserva como un único grupo `UNKNOWN_ST`, con IDs y counts exportados; no excluye la cohorte por falta de ST. Prefijos ST numéricos se normalizan al entero textual, siguiendo la generación Kleborate. Columnas biológicas opcionales solo se trasladan desde columnas explícitas configuradas de la tabla ST; deben estar completas y ser booleanas válidas, nunca se infieren de RGI o fenotipo.

Bolsas ≤50 independientes de etiquetas: prioridad opcional por `Drug Class` (requiere la columna raw), keywords carbapenem/beta-lactam/OXA/KPC/NDM/VIM/IMP/GES y muestreo por genoma con RNG estable. Guardar selección no implica inferir mecanismos. XGB conserva el máximo original **100 árboles**, depth=4 y lr=.05; early stopping interno puede seleccionar menos, nunca usa outer test. Arquitectura, focal/cosine y parámetros MIL se mantienen; sub-batch de genes=64 como en el flujo original, reducible por memoria.


In [ ]:
from pathlib import Path
import os, json, hashlib, re, random, platform, importlib.metadata, gc, ast, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

MOUNT_DRIVE = globals().get('MOUNT_DRIVE', True)
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        warnings.warn('No es Colab; Drive no se monta. Los defaults apuntan a Drive Colab.')
    else:
        drive.mount('/content/drive')

RUTA_BASE = '/content/drive/MyDrive/rgi_Klebsiella_pneumoniae_ertapenem'

CFG = dict(
    base=RUTA_BASE,
    raw_susceptible='/content/drive/MyDrive/rgi_Kp_ertapenem_Susceptible',
    raw_resistant='/content/drive/MyDrive/rgi_Kp_ertapenem_Resistente',
    allow_curated_fallback=False,  # Explicit opt-in; never mix original and curated directories.
    st_files=None, input_provenance=None,  # Explicit verified staging inputs; both enter identity.
    min_bytes=500, cutoff_ok=['Perfect','Strict'], min_seq_len=20,
    output_root=f'{RUTA_BASE}/validated_results',
    run_id='lineage_validation_20261013', run_training=False, resume=False,
    cohort_selection=None, cohort_selection_column='nombre_archivo_original',
    cohort_selection_filter_column=None, cohort_selection_filter_value=None,
    notebook_path=None,  # Attempt Colab metadata discovery; saved path is the only manual-path exception.
    seed=42, n_folds=5, inner_folds=5, max_genes=50, max_seq_len=512,
    prioritize_drug_class=True,
    keywords=['carbapenem', 'beta-lactam', 'OXA', 'KPC', 'NDM', 'VIM', 'IMP', 'GES'],
    model_name='InstaDeepAI/NTV3_100M_post', model_revision=None,
    allow_reviewed_remote_code=False, species_id=6,
    hidden_dim=128, num_heads=4, embed_dropout=.15,
    focal_alpha=.55, focal_gamma=2., ortho_lambda=.01,
    lr=1e-5, weight_decay=.01, max_epochs=20, patience=3, warmup_frac=.10,
    gene_sub_batch=64, bootstrap_B=2000, bootstrap_seed=1701, ci_level=.95,
    run_shap=False, shap_max_validation=64,
    biological_columns=[], aada2_column=None, carbapenemase_column=None,
    target_sensitivity=.95, use_amp=True, gradient_checkpointing=False,
    run_memory_smoke_test=False,
    xgb=dict(n_estimators=100, max_depth=4, learning_rate=.05,
             scale_pos_weight=1., objective='binary:logistic', eval_metric='auc',
             tree_method='hist', n_jobs=2, early_stopping_rounds=20),
)
RUN_OVERRIDES = globals().get('RUN_OVERRIDES', {})
assert isinstance(RUN_OVERRIDES, dict) and not set(RUN_OVERRIDES) - set(CFG), 'Unsupported RUN_OVERRIDES'
CFG.update(RUN_OVERRIDES)
# Execution selection is logged separately, never alters the identity of a fold.
EXECUTION_REQUEST = globals().get('EXECUTION_REQUEST', dict(mode='train' if CFG['run_training'] else 'preflight', scheme='both', fold=None))
assert set(EXECUTION_REQUEST) == {'mode','scheme','fold'}
assert EXECUTION_REQUEST['mode'] in {'preflight','smoke','train','analyze'}
assert EXECUTION_REQUEST['scheme'] in {'random','st_blocked','both'}
assert EXECUTION_REQUEST['fold'] is None or EXECUTION_REQUEST['fold'] in range(1,6)
assert CFG['run_training'] == (EXECUTION_REQUEST['mode'] != 'preflight')
assert bool(CFG['cohort_selection_filter_column']) == (CFG['cohort_selection_filter_value'] is not None)
assert not CFG['cohort_selection_filter_column'] or CFG['cohort_selection']
assert 1 <= CFG['max_genes'] <= 50
assert CFG['n_folds'] == 5 and CFG['inner_folds'] >= 2
assert CFG['max_epochs'] > 0 and CFG['patience'] > 0
assert CFG['bootstrap_B'] > 0 and 0 < CFG['ci_level'] < 1
assert 0 < CFG['target_sensitivity'] <= 1
assert not CFG['carbapenemase_column'] or CFG['aada2_column'], 'Cooccurrence requires aadA2 column'
assert not CFG['carbapenemase_column'] or CFG['carbapenemase_column'] != CFG['aada2_column']
assert re.fullmatch(r'[A-Za-z0-9_-]+', CFG['run_id'])
ROOT = Path(CFG['output_root']) / CFG['run_id']
ROOT.mkdir(parents=True, exist_ok=True)

def save_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)

def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

def file_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def versions():
    out = {'python': platform.python_version(), 'platform': platform.platform()}
    for p in ['numpy', 'pandas', 'scikit-learn', 'torch', 'transformers', 'xgboost', 'shap']:
        try: out[p] = importlib.metadata.version(p)
        except importlib.metadata.PackageNotFoundError: out[p] = 'not-installed'
    return out

def rgi_files(directory):
    directory = Path(directory)
    return sorted(p for p in directory.iterdir() if p.is_file() and p.suffix in {'.txt','.tsv'}) if directory.is_dir() else []

def discover_raw_sources():
    original = {0:Path(CFG['raw_susceptible']),1:Path(CFG['raw_resistant'])}
    available = all(rgi_files(path) for path in original.values())
    chosen, fallback = original, False
    if not available:
        assert CFG['allow_curated_fallback'], (
            'Original RGI directories unavailable/empty. Curated fallback requires explicit allow_curated_fallback=True.')
        chosen = {0:Path(CFG['base'])/'dataset_limpio'/'Susceptible',
                  1:Path(CFG['base'])/'dataset_limpio'/'Resistente'}
        assert all(rgi_files(path) for path in chosen.values()), 'Curated fallback also unavailable/empty'
        fallback = True
        warnings.warn('EXPLICIT CURATED FALLBACK: originals unavailable; copied folders may omit empty files/isolates. '
                      'This run cannot claim complete original cohort coverage. No mixing of sources.')
    inventory = [dict(path=str(path),label=label,source_directory=str(directory),filename=path.name)
                 for label,directory in chosen.items() for path in rgi_files(directory)]
    return inventory, dict(curated_fallback=fallback,
        chosen_directories={str(k):str(v) for k,v in chosen.items()},
        original_available={str(k):bool(rgi_files(v)) for k,v in original.items()})

RAW_INVENTORY, SOURCE_SELECTION = discover_raw_sources()
if CFG['st_files'] is not None:
    assert isinstance(CFG['st_files'], list) and CFG['st_files'], 'st_files must be an explicit nonempty list'
    ST_PATHS = [Path(path) for path in CFG['st_files']]
    assert len(set(ST_PATHS)) == len(ST_PATHS) and all(path.is_file() and path.suffix in {'.csv','.xlsx'} for path in ST_PATHS)
else:
    ST_PATHS = [path for path in [Path(CFG['base'])/'dataset_limpio'/'ST_table.csv',
                                 Path(CFG['base'])/'dataset_limpio'/'ST_table.xlsx'] if path.is_file()]
assert ST_PATHS, 'Missing explicit st_files or existing dataset_limpio/ST_table accession,ST.'
# Both existing representations are hashed and checked for equal canonical mappings below.
selection_paths = [Path(CFG['cohort_selection'])] if CFG['cohort_selection'] else []
provenance_paths = [Path(CFG['input_provenance'])] if CFG['input_provenance'] else []
input_hashes = {str(path):file_sha(path) for path in [*[r['path'] for r in RAW_INVENTORY],*ST_PATHS,*selection_paths,*provenance_paths]}
INPUTS = ROOT / 'inputs'
CFG['manifest'] = str(INPUTS/'manifest.csv')
CFG['sequences'] = [str(INPUTS/'rgi_sequences.csv')]

def saved_code_sources(path):
    notebook = json.loads(Path(path).read_text(encoding='utf-8'))
    assert notebook.get('nbformat') == 4, 'Expected saved .ipynb'
    return [''.join(cell['source']) for cell in notebook['cells'] if cell['cell_type'] == 'code']

def implementation_hash(path):
    # Code only: execution counts, outputs and markdown edits do not invalidate folds.
    # Cell boundaries and exact source are retained; any code change changes the hash.
    return digest(saved_code_sources(path))

def infer_saved_colab_notebook():
    # Colab title is metadata, not a reliable Drive path. Check only existing candidates
    # and verify their code against the current editor; never invent a saved path.
    try:
        from google.colab import _message
    except ImportError:
        return None
    try:
        reply = _message.blocking_request('get_ipynb', timeout_sec=10)
        live = reply.get('ipynb', {}) if isinstance(reply,dict) else {}
        name = live.get('metadata', {}).get('colab', {}).get('name')
        if not name or Path(name).name != name: return None
        live_code = [''.join(c['source']) for c in live.get('cells',[]) if c['cell_type'] == 'code']
        candidates = [Path('/content/drive/MyDrive/Colab Notebooks')/name,
                      Path('/content/drive/MyDrive')/name, Path(CFG['base'])/name]
        matches = [str(path) for path in candidates if path.is_file() and saved_code_sources(path) == live_code]
        return matches[0] if len(matches) == 1 else None
    except Exception as error:
        warnings.warn(f'Colab saved-path discovery unavailable: {type(error).__name__}; no path guessed.')
        return None

if CFG['run_training'] or CFG['resume']:
    if CFG['notebook_path'] is None: CFG['notebook_path'] = infer_saved_colab_notebook()
    assert CFG['notebook_path'] and Path(CFG['notebook_path']).is_file(), (
        'Colab could not uniquely locate the saved notebook. Only exception: set notebook_path to its existing saved Drive path, save and Run all.')
IMPLEMENTATION_HASH = implementation_hash(CFG['notebook_path']) if CFG['notebook_path'] else None
VERIFIED_INPUT_PROVENANCE = None
if CFG['input_provenance']:
    VERIFIED_INPUT_PROVENANCE = json.loads(Path(CFG['input_provenance']).read_text(encoding='utf-8'))
    assert file_sha(CFG['input_provenance']) == input_hashes[CFG['input_provenance']], 'Staging manifest changed during read'
    proof = VERIFIED_INPUT_PROVENANCE
    assert proof['schema'] == 'verified_lineage_inputs_v1' and proof['status'] == 'complete'
    assert IMPLEMENTATION_HASH and proof['notebook_code_sha256'] == IMPLEMENTATION_HASH, 'Staging notebook code differs'
    assert proof['runner_sha256'] == globals().get('CLI_RUNNER_SHA256'), 'Staging runner differs; use the verified CLI runner'
    assert not SOURCE_SELECTION['curated_fallback'], 'Verified staging must use explicitly configured sources'
    assert proof['raw_sources'] == {str(k):str(v) for k,v in {0:Path(CFG['raw_susceptible']),1:Path(CFG['raw_resistant'])}.items()}
    assert proof['st_files'] == [str(path) for path in ST_PATHS] and proof['cohort_selection'] == CFG['cohort_selection']
    assert CFG['cohort_selection_column'] == 'nombre_archivo_original'
    assert CFG['cohort_selection_filter_column'] == 'split' and CFG['cohort_selection_filter_value'] == '5Fold'
    targets = [entry['target_path'] for entry in proof['entries']]
    assert len(targets) == len(set(targets)) == proof['expected']['total'] == 3708
    assert set(targets) == {entry['path'] for entry in RAW_INVENTORY}, 'Staging inventory differs'
    assert sum(entry['alias_applied'] for entry in proof['entries']) == proof['expected']['aliases'] == 6
    assert proof['expected']['exact'] == 3702 and proof['expected']['label_counts'] == {'0':2259,'1':1449}
    assert {str(label):sum(entry['label'] == label for entry in proof['entries']) for label in [0,1]} == proof['expected']['label_counts']
    stage_root = Path(proof['stage_root']).resolve()
    assert set(targets) | set(map(str,ST_PATHS)) | {CFG['cohort_selection']} <= set(proof['artifacts'])
    for path,sha in proof['artifacts'].items():
        assert Path(path).resolve().is_relative_to(stage_root) and re.fullmatch(r'[0-9a-f]{64}',sha)
        if path not in input_hashes: input_hashes[path] = file_sha(path)
        assert input_hashes[path] == sha, f'Verified staged artifact changed: {path}'
    assert proof['source_hashes'] and all(re.fullmatch(r'[0-9a-f]{64}',sha) for sha in proof['source_hashes'].values())
    for entry in proof['entries']:
        assert entry['source_sha256'] == proof['source_hashes'][entry['source_path']] == entry['target_sha256'] == input_hashes[entry['target_path']]
    # Original Drive hashes are measured by staging and retained here as provenance.
    # We verify local immutable snapshots; we do not claim to rehash live originals on every fold.
# Resolve metadata only when training; preflight neither queries the hub nor downloads models.
# On resume keep the previously resolved commit even if the remote main branch advanced.
if CFG['run_training']:
    assert CFG['allow_reviewed_remote_code'], 'Explicit consent to reviewed NTv3 remote code is required for training'
    existing_identity = json.loads((ROOT/'identity.json').read_text()) if (ROOT/'identity.json').exists() else None
    if CFG['model_revision'] is None and existing_identity is not None and CFG['resume']:
        old = existing_identity['config']
        assert old['model_name'] == CFG['model_name'], 'Foundation repository changed'
        CFG['model_revision'] = old['model_revision']
    assert EXECUTION_REQUEST['mode'] != 'analyze' or CFG['model_revision'], 'Analysis needs an existing foundation lock; use resume'
    if CFG['model_revision'] is None:
        from huggingface_hub import HfApi
        CFG['model_revision'] = HfApi().model_info(CFG['model_name'],revision='main').sha
    assert re.fullmatch(r'[0-9a-f]{40}', CFG['model_revision'] or ''), 'Hub did not return immutable commit SHA'
identity_cfg = {k:v for k,v in CFG.items() if k not in ['run_training','resume']}
identity_overrides = {k:v for k,v in RUN_OVERRIDES.items() if k not in ['run_training','resume']}
identity = dict(config=identity_cfg, inputs=input_hashes, implementation_sha256=IMPLEMENTATION_HASH,
                run_overrides=identity_overrides, run_overrides_sha256=digest(identity_overrides),
                runtime_script_sha256=globals().get('CLI_RUNNER_SHA256'),
                raw_inventory=RAW_INVENTORY, source_selection=SOURCE_SELECTION,
                verified_input_provenance=VERIFIED_INPUT_PROVENANCE)


def verify_saved_execution(stage_marker):
    assert IMPLEMENTATION_HASH is not None, 'Train/resume requires a saved notebook'
    assert {k:v for k,v in CFG.items() if k not in ['run_training','resume']} == identity['config'], (
        'Runtime configuration changed after fingerprint; restart Run all.')
    sources = saved_code_sources(CFG['notebook_path'])
    assert implementation_hash(CFG['notebook_path']) == IMPLEMENTATION_HASH, (
        'Saved code changed since fingerprint; restart Run all with a new run_id.')
    assert {k:v for k,v in RUN_OVERRIDES.items() if k not in ['run_training','resume']} == identity['run_overrides'], 'Overrides changed after fingerprint'
    # Find the implementation cell, not the verifier's call-site string in an earlier cell.
    stop = next(i for i, source in enumerate(sources) if any(line.lstrip().startswith(stage_marker) for line in source.splitlines()))
    if globals().get('CLI_VERIFY_EXECUTION') is not None:
        CLI_VERIFY_EXECUTION(sources, stop, IMPLEMENTATION_HASH)
        assert file_sha(CLI_RUNNER_PATH) == identity['runtime_script_sha256'], 'Runner source changed'
        return
    # IPython/Colab records the currently executing cell before its body executes.
    # Require an ordered full-code match, not merely function names or a manual version.
    ip = get_ipython()
    assert ip is not None, 'Execute train/resume in IPython/Colab from the saved notebook'
    history = ip.history_manager.input_hist_raw
    expected = [ast.dump(ast.parse(source), include_attributes=False) for source in sources[:stop+1]]
    actual = []
    for source in history:
        try: actual.append(ast.dump(ast.parse(source), include_attributes=False))
        except SyntaxError: continue  # Other interactive/magic cells are not protocol code.
    # Latest occurrences prevent a stale old matching run from hiding an edited cell.
    cursor = len(actual)
    for signature in reversed(expected):
        matches = [i for i in range(cursor) if actual[i] == signature]
        assert matches, 'Executed cells differ from saved notebook; save and restart Run all.'
        cursor = matches[-1]
    # No mismatching protocol definitions may have been re-executed after this prefix.
    relevant_names = {'CFG'} | {node.name for source in sources[:stop+1]
        for node in ast.walk(ast.parse(source)) if isinstance(node, (ast.FunctionDef, ast.ClassDef))}
    for source in history:
        try: tree = ast.parse(source)
        except SyntaxError: continue
        touches_protocol = any(
            isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in relevant_names
            or isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store) and node.id in relevant_names
            for node in ast.walk(tree))
        if touches_protocol:
            assert ast.dump(tree, include_attributes=False) in expected, (
                'Unsaved/redefined protocol code in this runtime; restart and Run all.')

FINGERPRINT = digest(identity)
if (ROOT / 'identity.json').exists():
    assert json.loads((ROOT / 'identity.json').read_text()) == identity, (
        'Run directory has incompatible config/data/code; use a new run_id.')
else:
    assert not any(ROOT.iterdir()), 'Existing unrecognized directory: choose a new run_id.'
    save_json(ROOT / 'identity.json', identity)
save_json(ROOT / 'environment_preflight.json', versions())
save_json(ROOT / 'source_selection.json', SOURCE_SELECTION)
if VERIFIED_INPUT_PROVENANCE is not None: save_json(ROOT / 'verified_input_provenance.json', VERIFIED_INPUT_PROVENANCE)
save_json(ROOT / 'run_overrides.json', dict(overrides=identity_overrides, sha256=digest(identity_overrides)))
print('Effective CFG:', json.dumps(CFG, sort_keys=True))
print('Execution request:', EXECUTION_REQUEST, 'implementation SHA256:', IMPLEMENTATION_HASH)
if CFG['run_training']: save_json(ROOT / 'foundation_lock.json',dict(repository=CFG['model_name'],commit=CFG['model_revision']))
print('CPU preflight; training requested:', CFG['run_training'], 'output:', ROOT)

In [ ]:
# These inputs are generated exclusively from the discovered raw inventory and ST files.
# Never from old folds, OOF, train_por_cepa.csv or label-dependent sequence filtering.
INPUTS.mkdir(exist_ok=True)

def canonical_id(value):
    assert pd.notna(value), 'Null identifier'
    text = str(value).strip()
    assert text, 'Empty identifier'
    # Verified historical naming: a versioned accession embedded in a filename,
    # otherwise only terminal .txt/.tsv and then terminal _rgi removal.
    accessions = re.findall(r'GC[AF]_\d+\.\d+', text)
    assert len(set(accessions)) <= 1, f'Ambiguous accession identifier: {text}'
    if accessions: return accessions[0], 'single_distinct_versioned_GCA_GCF_accession'
    trimmed = re.sub(r'\.(txt|tsv)$', '', text)
    trimmed = re.sub(r'_rgi$', '', trimmed)
    assert trimmed, f'Empty canonical identifier: {text}'
    return trimmed, 'terminal_txt_tsv_then_rgi_suffix'

def clean_st(values):
    st = values.astype('string').fillna('').str.strip()
    unknown = st.str.lower().isin(['','na','nan','unknown','unk','nd','n/a','-','?','st-'])
    # Kleborate generation strips ST; numeric ST prefixes denote the same known integer.
    st = st.str.replace(r'^ST(?=\d+$)', '', regex=True)
    return st.mask(unknown,'UNKNOWN_ST')

bool_columns = list(dict.fromkeys(CFG['biological_columns'] +
    ([CFG['aada2_column']] if CFG['aada2_column'] else []) +
    ([CFG['carbapenemase_column']] if CFG['carbapenemase_column'] else [])))
assert not set(bool_columns) & {'genome_id','accession','label','ST','raw_genome_id'}

def load_st_table(path):
    # Exact schema written by st_records.append({'accession':acc_out,'ST':st}).
    if path.suffix == '.xlsx': table = pd.read_excel(path,dtype='string')
    else:
        try: table = pd.read_csv(path,dtype='string')
        except pd.errors.EmptyDataError: table = pd.DataFrame(columns=['accession','ST'])
    table.columns = [str(c).strip() for c in table.columns]
    assert not table.columns.duplicated().any(), 'Duplicated ST columns'
    if table.empty and not {'accession','ST'} <= set(table.columns):
        table = pd.DataFrame(columns=['accession','ST'])  # Historical empty st_records file.
    assert {'accession','ST'} <= set(table.columns), f'ST schema must be accession,ST: {path}'
    table['st_raw_accession'] = table['accession']
    assert file_sha(path) == input_hashes[str(path)], 'ST data changed after fingerprint'
    table['genome_id'] = table['accession'].map(lambda x:canonical_id(x)[0]).astype('string')
    table['st_id_rule'] = table['accession'].map(lambda x:canonical_id(x)[1])
    assert not table.genome_id.duplicated().any(), f'Ambiguous/duplicate canonical ST mapping: {path}'
    table['ST'] = clean_st(table['ST'])
    for column in bool_columns:
        assert column in table, f'Optional annotation explicitly requested but absent in ST table: {column}'
        values = table[column].astype('string').str.strip().str.lower()
        assert values.notna().all() and values.isin(['0','1','true','false']).all(), f'Invalid boolean {column}'
        table[column] = values.map({'0':0,'1':1,'true':1,'false':0}).astype(int)
    return table

st_tables = [load_st_table(path) for path in ST_PATHS]
df_st = st_tables[0]
for other in st_tables[1:]:
    columns = ['genome_id','ST',*bool_columns]
    pd.testing.assert_frame_equal(df_st[columns].sort_values('genome_id').reset_index(drop=True),
        other[columns].sort_values('genome_id').reset_index(drop=True),check_dtype=False,
        obj='CSV/XLSX canonical ST mapping (conflicts must be curated, never silently preferred)')

inventory = pd.DataFrame(RAW_INVENTORY)
inventory['genome_id'] = inventory.filename.map(lambda x:canonical_id(x)[0])
inventory['filename_id_rule'] = inventory.filename.map(lambda x:canonical_id(x)[1])
assert not inventory.genome_id.duplicated().any(), (
    'More than one raw file maps to an isolate or S/R conflict: ambiguous cohort; curate explicitly.')
inventory['source_file'] = inventory['path']
inventory['raw_genome_id'] = inventory.filename
inventory['source_sha256'] = inventory.path.map(lambda x:input_hashes[x])
inventory.to_csv(INPUTS/'raw_inventory.csv',index=False)

if CFG['cohort_selection']:
    selection_path = Path(CFG['cohort_selection'])
    assert file_sha(selection_path) == input_hashes[str(selection_path)], 'Cohort selection changed'
    selection = pd.read_csv(selection_path, dtype='string', keep_default_na=False)
    column = CFG['cohort_selection_column']
    assert column in selection, f'Missing explicit provenance column: {column}'
    if CFG['cohort_selection_filter_column']:
        filter_column = CFG['cohort_selection_filter_column']
        assert filter_column in selection, f'Missing explicit cohort filter: {filter_column}'
        selection = selection.loc[selection[filter_column].eq(CFG['cohort_selection_filter_value'])].copy()
    assert len(selection), 'Empty explicit historical selection'
    # Exact original filename bridge only: no decimals, underscore conversions, GCA/GCF or version equivalence.
    assert selection[column].str.strip().eq(selection[column]).all() and selection[column].ne('').all()
    assert selection[column].map(lambda value: Path(value).name == value).all(), 'Expected original filenames, not inferred paths/IDs'
    assert not selection[column].duplicated().any(), 'Ambiguous duplicate historical provenance filenames'
    assert not inventory.filename.duplicated().any(), 'Ambiguous raw filename bridge'
    bridge = selection.merge(inventory[['filename','genome_id','source_file','source_sha256','label']],
        left_on=column, right_on='filename', how='left', validate='one_to_one', indicator=True, suffixes=('_historical',''))
    bridge.to_csv(INPUTS/'historical_cohort_bridge.csv',index=False)
    assert bridge['_merge'].eq('both').all(), 'Unmatched historical filenames; inspect bridge, no equivalence inferred'
    for phenotype_column in ['origen_dataset','cepasv2_phenotype']:
        if phenotype_column in bridge:
            values = bridge[phenotype_column]
            known = values.ne('')
            historical_labels = values.loc[known].map({'Resistant':1,'Susceptible':0})
            assert historical_labels.notna().all() and historical_labels.eq(bridge.loc[known,'label']).all(), 'Historical/raw phenotype conflict'
            # Missing historical annotations remain missing; raw S/R directories supply labels.
    inventory.loc[~inventory.genome_id.isin(bridge.genome_id)].to_csv(INPUTS/'outside_historical_selection.csv',index=False)
    inventory = inventory.loc[inventory.genome_id.isin(bridge.genome_id)].copy()

records, file_audit, row_exclusions = [], [], []
for item in inventory.itertuples():
    path = Path(item.path)
    assert file_sha(path) == input_hashes[str(path)], 'Raw data changed after fingerprint'
    audit = dict(genome_id=item.genome_id,source_file=str(path),label=int(item.label),
                 bytes=path.stat().st_size,raw_rows=0,valid_hits=0,input_status='pending')
    if audit['bytes'] <= CFG['min_bytes']:
        audit['input_status'] = 'file_size_le_min_bytes'
        file_audit.append(audit); continue
    try:
        raw = pd.read_csv(path,sep='\t',dtype='string')
    except pd.errors.EmptyDataError:
        audit['input_status'] = 'empty_RGI_file'
        file_audit.append(audit); continue
    except (pd.errors.ParserError,UnicodeError) as error:
        audit['input_status'] = 'parse_error'
        file_audit.append(audit)
        pd.DataFrame(file_audit).to_csv(INPUTS/'raw_file_audit.csv',index=False)
        raise ValueError(f'Cannot parse RGI TSV {path}; no silent file omission') from error
    assert {'Cut_Off','Predicted_DNA'} <= set(raw.columns), f'RGI schema mismatch: {path}'
    if CFG['prioritize_drug_class']:
        assert 'Drug Class' in raw.columns, f'Raw Drug Class missing: {path}; disable priority explicitly if intentional'
    audit['raw_rows'] = len(raw)
    for offset, row in enumerate(raw.to_dict('records'),2):
        cutoff = '' if pd.isna(row['Cut_Off']) else str(row['Cut_Off'])
        dna = '' if pd.isna(row['Predicted_DNA']) else str(row['Predicted_DNA']).strip().upper()
        reason = None
        if cutoff not in CFG['cutoff_ok']: reason = 'Cut_Off_not_Perfect_or_Strict'
        elif not dna: reason = 'empty_Predicted_DNA'
        elif len(dna) <= CFG['min_seq_len']: reason = 'sequence_length_le_min_seq_len'
        if reason:
            row_exclusions.append(dict(genome_id=item.genome_id,source_file=str(path),source_row=offset,
                reason=reason,Cut_Off=str(cutoff),sequence_length=len(dna)))
            continue
        assert re.fullmatch(r'[ACGTRYSWKMBDHVN]+',dna), f'Invalid DNA in {path}, parsed record {offset}'
        drug_class = row.get('Drug Class','')
        records.append(dict(genome_id=item.genome_id,raw_genome_id=item.filename,label=int(item.label),
            sequence=dna,**{'Drug Class':'' if pd.isna(drug_class) else str(drug_class)},
            Cut_Off=str(cutoff),archivo=item.filename,source_file=str(path),source_row=offset))
        audit['valid_hits'] += 1
    audit['input_status'] = 'valid_RGI_bag' if audit['valid_hits'] else 'no_valid_Perfect_Strict_DNA_hits'
    file_audit.append(audit)

file_audit = pd.DataFrame(file_audit)
file_audit.to_csv(INPUTS/'raw_file_audit.csv',index=False)
pd.DataFrame(row_exclusions,columns=['genome_id','source_file','source_row','reason','Cut_Off','sequence_length']).to_csv(
    INPUTS/'raw_row_exclusions.csv',index=False)
# Every selected raw file is in the manifest, including empty/quality-excluded bags; full inventory is retained.
manifest = inventory.merge(file_audit[['genome_id','input_status','valid_hits']],on='genome_id',validate='one_to_one')
manifest = manifest.merge(df_st[['genome_id','ST','st_raw_accession','st_id_rule',*bool_columns]],
    on='genome_id',how='left',validate='one_to_one',indicator='st_join')
manifest['st_match_status'] = np.where(manifest.st_join.eq('left_only'),'unmatched',
                                     np.where(manifest.ST.eq('UNKNOWN_ST').fillna(False),'matched_unknown','matched_known'))
manifest['ST'] = manifest.ST.fillna('UNKNOWN_ST')
manifest.drop(columns=['path','st_join'],inplace=True)
manifest.sort_values('genome_id').to_csv(CFG['manifest'],index=False)
sequence_columns = ['genome_id','raw_genome_id','label','sequence','Drug Class','Cut_Off','archivo','source_file','source_row']
pd.DataFrame(records,columns=sequence_columns).to_csv(CFG['sequences'][0],index=False)
manifest.loc[manifest.ST.eq('UNKNOWN_ST')].to_csv(INPUTS/'unknown_ST_isolates.csv',index=False)
df_st.loc[~df_st.genome_id.isin(manifest.genome_id)].to_csv(INPUTS/'ST_outside_raw_cohort.csv',index=False)
save_json(INPUTS/'preparation_audit.json',dict(raw_files=len(inventory),valid_sequences=len(records),
    empty_or_filtered_isolates=int(manifest.valid_hits.eq(0).sum()),
    unknown_ST=int(manifest.ST.eq('UNKNOWN_ST').sum()),unmatched_ST=int(manifest.st_match_status.eq('unmatched').sum()),
    curated_fallback=SOURCE_SELECTION['curated_fallback'],label_source='explicit source directories: S=0, R=1',
    quality_filter=dict(file_bytes_gt=CFG['min_bytes'],cutoff_ok=CFG['cutoff_ok'],DNA_length_gt=CFG['min_seq_len']),
    label_dependent_sequence_filter=False,sequence_duplicates='distinct raw records/loci retained',
    optional_annotations=bool_columns))
print('Prepared raw explicit manifest:',len(manifest),'isolates;',len(records),'sequences;',
      int(manifest.ST.eq('UNKNOWN_ST').sum()),'unknown ST; inputs:',INPUTS)

In [ ]:
def normalize_id(series):
    assert series.notna().all(), 'Null genome IDs'
    values = series.map(lambda value:canonical_id(value)[0])
    assert values.ne('').all(), 'Empty genome IDs'
    return values

manifest = pd.read_csv(CFG['manifest'], dtype={'genome_id': 'string', 'ST': 'string'})
assert {'genome_id','label','ST'} <= set(manifest.columns)
assert {'raw_genome_id','source_file','input_status'} <= set(manifest.columns)
manifest['genome_id'] = normalize_id(manifest['genome_id'])
assert not manifest['genome_id'].duplicated().any(), 'Duplicate/colliding manifest IDs'
labels = pd.to_numeric(manifest['label'], errors='raise')
assert labels.notna().all() and labels.isin([0,1]).all(), 'Labels must be 0/1'
manifest['label'] = labels.astype(int)
st = manifest['ST'].fillna('').str.strip()
manifest['ST'] = clean_st(st)

bool_columns = list(dict.fromkeys(CFG['biological_columns'] +
               ([CFG['aada2_column']] if CFG['aada2_column'] else []) +
               ([CFG['carbapenemase_column']] if CFG['carbapenemase_column'] else [])))
assert not set(bool_columns) & {'genome_id','label','ST','raw_genome_id'}
for column in bool_columns:
    assert column in manifest, f'Missing explicit biological column: {column}'
    values = manifest[column].astype(str).str.strip().str.lower()
    assert values.isin(['0','1','true','false']).all(), f'Invalid/null boolean: {column}'
    manifest[column] = values.map({'0':0,'1':1,'true':1,'false':0}).astype(int)

parts = []
for path in CFG['sequences']:
    part = pd.read_csv(path, dtype={'genome_id':'string', 'sequence':'string'})
    assert {'genome_id','sequence'} <= set(part.columns)
    assert {'source_file','source_row','raw_genome_id'} <= set(part.columns), 'Prepared raw provenance required'
    part['genome_id'] = normalize_id(part['genome_id'])
    parts.append(part)
rgi = pd.concat(parts, ignore_index=True)
# Repeated normalized ID across RGI rows is expected, distinct raw aliases are not.
aliases = rgi.groupby('genome_id')['raw_genome_id'].nunique()
assert aliases.le(1).all(), 'Normalization collisions within RGI'
rgi['sequence'] = rgi['sequence'].fillna('').str.strip().str.upper()
invalid = rgi['sequence'].ne('') & ~rgi['sequence'].str.fullmatch(r'[ACGTRYSWKMBDHVN]+')
assert not invalid.any(), 'Invalid nucleotide sequence; inspect RGI export (no silent repair)'
external = rgi.loc[~rgi.genome_id.isin(manifest.genome_id)]
external.to_csv(ROOT / 'rgi_outside_manifest.csv', index=False)
rgi = rgi.loc[rgi.genome_id.isin(manifest.genome_id)].copy()
# Audit labels only; do not select sequences according to phenotype.
if 'label' in rgi:
    check = rgi[['genome_id','label']].merge(
        manifest[['genome_id','label']], on='genome_id', validate='many_to_one',
        suffixes=('_rgi','_manifest'))
    assert pd.to_numeric(check.label_rgi, errors='raise').eq(check.label_manifest).all()
rgi.loc[rgi.sequence.eq('')].to_csv(ROOT / 'excluded_sequence_rows.csv', index=False)
rgi = rgi.loc[rgi.sequence.ne('')].copy()
# Distinct raw RGI records/loci may encode identical sequences, regardless of label.
# Never delete shared sequences globally; each unique source row remains an instance.
assert not rgi.duplicated(['source_file','source_row']).any(), 'Duplicate source record'
if CFG['prioritize_drug_class']:
    assert 'Drug Class' in rgi, 'Disable priority explicitly if Drug Class is unavailable'

bag_rows, selected = [], []
for gid, g in rgi.groupby('genome_id', sort=True):
    g = g.sort_values(['sequence','source_file','source_row']).copy()
    seed = int(hashlib.sha256(f"{CFG['seed']}:{gid}".encode()).hexdigest()[:16], 16)
    rng = np.random.default_rng(seed)
    candidates = g.index.to_numpy()
    if len(g) > CFG['max_genes']:
        if CFG['prioritize_drug_class']:
            pattern = '|'.join(re.escape(k) for k in CFG['keywords'])
            relevant = g['Drug Class'].fillna('').str.contains(pattern, case=False, regex=True)
            rel, other = g.index[relevant].to_numpy(), g.index[~relevant].to_numpy()
            if len(rel) >= CFG['max_genes']:
                candidates = rng.choice(rel, CFG['max_genes'], replace=False)
            else:
                candidates = np.concatenate([rel, rng.choice(other, CFG['max_genes']-len(rel), replace=False)])
        else:
            candidates = rng.choice(candidates, CFG['max_genes'], replace=False)
    chosen = g.loc[candidates].sort_values(['sequence','source_file','source_row'])
    bag_rows.append({'genome_id':gid, 'sequences':chosen.sequence.tolist(), 'bag_size':len(chosen)})
    for order, (_, row) in enumerate(chosen.iterrows()):
        selected.append({'genome_id':gid, 'bag_order':order,
            'source_file':row.source_file, 'source_row':int(row.source_row),
            'sequence_sha256':hashlib.sha256(row.sequence.encode()).hexdigest(),
            'sequence':row.sequence, 'Drug Class':str(row.get('Drug Class',''))})
assert bag_rows, 'No nonempty bags'
bag_table = pd.DataFrame(bag_rows)
joined = manifest.merge(bag_table, on='genome_id', how='left', validate='one_to_one', indicator=True)
excluded = joined.loc[joined['_merge'].eq('left_only'), ['genome_id','label','ST','source_file','input_status']].copy()
excluded['reason'] = excluded['input_status']
excluded.to_csv(ROOT / 'excluded_genomes.csv', index=False)
assert not CFG['cohort_selection'] or excluded.empty, 'Historical selection has ineligible bags: cannot claim identical cohort; inspect excluded_genomes.csv'
cohort = joined.loc[joined['_merge'].eq('both')].drop(columns='_merge').sort_values('genome_id').reset_index(drop=True)
assert cohort.label.nunique() == 2
assert cohort.bag_size.between(1, CFG['max_genes']).all()
pd.DataFrame(selected).to_csv(ROOT / 'bag_selection.csv', index=False)
cohort.drop(columns='sequences').to_csv(ROOT / 'cohort.csv', index=False)
save_json(ROOT / 'audit.json', dict(manifest_n=len(manifest), included_n=len(cohort),
    excluded_n=len(excluded), external_rgi_rows=len(external), unknown_st_n=int(cohort.ST.eq('UNKNOWN_ST').sum()),
    label_counts={str(k):int(v) for k,v in cohort.label.value_counts().items()},
    st_counts={str(k):int(v) for k,v in cohort.ST.value_counts().items()}))
print('Included:', len(cohort), 'excluded:', len(excluded), 'ST groups:', cohort.ST.nunique())

## 2. Particiones anidadas y preflight

Outer random: StratifiedKFold; outer ST-blocked: StratifiedGroupKFold. Inner usa el mismo tipo de splitter **solo sobre outer train**; el primer split determinista es la validación interna, sin buscar el split con mejor resultado. Las heurísticas de estratificación por grupos no garantizan balance: si una partición no tiene ambas clases o grupos suficientes, se aborta. Nunca hay fallback aleatorio para ST-blocked. El esquema random permite linajes compartidos intencionalmente; se reporta su solapamiento.

Preflight no prueba compatibilidad real del modelo remoto ni memoria GPU. El entrenamiento requiere consentimiento explícito para el código remoto revisado. HfApi resuelve automáticamente el SHA de `main` solo al entrenar, lo bloquea en identidad/foundation_lock y lo reutiliza al reanudar; no se pide inventar un SHA. Preflight no consulta el hub ni descarga modelo/tokenizador. No se cargan checkpoints supervisados previos; cada fold empieza desde foundation. Reanudar significa omitir **folds completos verificados**, no continuar un optimizador interrumpido.

In [ ]:
def split_generator(frame, blocked, n, seed):
    assert frame.label.value_counts().min() >= n, 'Insufficient class counts for stratification'
    if blocked:
        assert frame.ST.nunique() >= n, 'Insufficient ST groups'
        return StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=seed).split(
            frame, frame.label, frame.ST)
    return StratifiedKFold(n_splits=n, shuffle=True, random_state=seed).split(frame, frame.label)

plans, fold_rows, overlap_rows = {}, [], []
for scheme, blocked in [('random',False), ('st_blocked',True)]:
    seen = []
    for fold, (train_idx, test_idx) in enumerate(split_generator(cohort, blocked, 5, CFG['seed']), 1):
        outer_train = cohort.iloc[train_idx].reset_index(drop=True)
        it, iv = next(split_generator(outer_train, blocked, CFG['inner_folds'], CFG['seed']+fold))
        indices = {'inner_train': train_idx[it], 'inner_val': train_idx[iv], 'outer_test':test_idx}
        frames = {role:cohort.iloc[idx] for role,idx in indices.items()}
        for role, f in frames.items():
            assert f.label.nunique() == 2, f'{scheme}/{fold}/{role}: one class'
            for gid in f.genome_id:
                fold_rows.append(dict(scheme=scheme, fold=fold, role=role, genome_id=gid))
        roles = list(frames)
        for i, a in enumerate(roles):
            for b in roles[i+1:]:
                assert not set(frames[a].genome_id) & set(frames[b].genome_id)
                shared = set(frames[a].ST) & set(frames[b].ST)
                if blocked: assert not shared, f'ST leakage: {scheme}/{fold}/{a}/{b}'
                overlap_rows.append(dict(scheme=scheme, fold=fold, roles=f'{a}:{b}', shared_st=len(shared)))
        assert set(frames['inner_train'].genome_id) | set(frames['inner_val'].genome_id) == set(outer_train.genome_id)
        seen.extend(frames['outer_test'].genome_id)
        plans[(scheme,fold)] = indices
    assert len(seen) == len(cohort) and len(set(seen)) == len(cohort)
    assert set(seen) == set(cohort.genome_id)
folds = pd.DataFrame(fold_rows).merge(cohort[['genome_id','label','ST']], on='genome_id', validate='many_to_one')
folds.to_csv(ROOT / 'fold_assignments.csv', index=False)
pd.DataFrame(overlap_rows).to_csv(ROOT / 'split_overlap_audit.csv', index=False)
print(folds.groupby(['scheme','fold','role','label']).size().unstack(fill_value=0))
print('Preflight completed; no training or foundation download performed by this cell.')

## 3. NTv3 y entrenamiento opcional

Una bolsa por paso y sub-batches de genes; AMP FP16 con GradScaler opcional en CUDA, FP32 en CPU. Se desescala antes del clipping; el scheduler no avanza si el scaler omite el paso. Gradient checkpointing es opcional: se exige soporte declarado por NTv3, API no reentrante y activación verificable; no se congela ni se usa `no_grad` sobre el encoder durante train. El smoke test opcional hace forward/backward con `max_genes` y padding a `max_seq_len`, solo con inner train y sin actualizar pesos, y exige gradientes no nulos del backbone. Puede repetir secuencias para alcanzar el máximo; no garantiza todos los patrones de memoria posibles. Se ejecuta antes de entrenar cada fold no completado. Checkpoints se cargan en CPU y se liberan junto con optimizer/scheduler/scaler antes de extraer embeddings. El modelo completo se ajusta solo con inner train. Cosine warmup, AdamW y selección por ROC AUC inner val; se conserva el primer máximo. No se reentrena con outer train después de seleccionar: MIL y XGB comparten inner train/val. Congelar embeddings significa evaluación `eval()` + `no_grad()` del checkpoint seleccionado; dropout está apagado. XGBoost ≥2.0 usa early stopping en el constructor, y versiones antiguas la firma de `fit`; incompatibilidades abortan, no se omite early stopping silenciosamente.

La longitud 512 trunca secuencias según el tokenizador; la selección de bolsa y hashes queda guardada. Puede haber fallos de memoria por el grafo del backbone: reducir sub-batch no elimina todos los activations de la bolsa. No se declara entrenamiento GPU validado.

In [ ]:
if CFG['run_training'] and EXECUTION_REQUEST['mode'] != 'analyze':
    verify_saved_execution('def seed_all(seed):')
    assert re.fullmatch(r'[0-9a-f]{40}', CFG['model_revision'] or ''), 'Set immutable model commit SHA'
    assert CFG['allow_reviewed_remote_code'], 'Review model remote code before opting in'
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import xgboost as xgb
    import inspect
    from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    AMP_ENABLED = bool(CFG['use_amp'] and device.type == 'cuda')
    from contextlib import nullcontext

    def amp_context():
        if not AMP_ENABLED: return nullcontext()
        if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
            return torch.amp.autocast('cuda', dtype=torch.float16)
        return torch.cuda.amp.autocast(dtype=torch.float16)

    def make_scaler():
        if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
            return torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
        return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

    current_versions = versions()
    version_path = ROOT / 'training_environment.json'
    if version_path.exists():
        assert json.loads(version_path.read_text()) == current_versions, 'Environment changed; use a new run_id'
    else: save_json(version_path, current_versions)
    tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'], revision=CFG['model_revision'],
                                             trust_remote_code=True)
    tokenizer.save_pretrained(ROOT / 'tokenizer')

    def seed_all(seed):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        # Best effort: remote kernels/GPU can remain nondeterministic; seeds are not a bitwise guarantee.

    class NTv3MultiHeadMIL(nn.Module):
        def __init__(self):
            super().__init__()
            self.ntv3 = AutoModel.from_pretrained(CFG['model_name'], revision=CFG['model_revision'],
                                                 trust_remote_code=True)
            if CFG['gradient_checkpointing']:
                assert getattr(self.ntv3, 'supports_gradient_checkpointing', False), (
                    'This NTv3 commit does not advertise gradient checkpointing support')
                enable = getattr(self.ntv3, 'gradient_checkpointing_enable', None)
                assert callable(enable), 'No gradient_checkpointing_enable API'
                assert 'gradient_checkpointing_kwargs' in inspect.signature(enable).parameters, (
                    'Need explicit non-reentrant checkpoint API for this NTv3 commit')
                enable(gradient_checkpointing_kwargs={'use_reentrant': False})
                assert getattr(self.ntv3, 'is_gradient_checkpointing', False), 'Checkpointing not enabled'
                if hasattr(self.ntv3.config, 'use_cache'): self.ntv3.config.use_cache = False
            self.layer_norm = nn.LayerNorm(768)
            self.embed_dropout = nn.Dropout(CFG['embed_dropout'])
            self.attention_heads = nn.ModuleList([nn.Sequential(
                nn.Linear(768, CFG['hidden_dim']), nn.Tanh(), nn.Dropout(.1),
                nn.Linear(CFG['hidden_dim'],1)) for _ in range(CFG['num_heads'])])
            self.projection = nn.Linear(768 * CFG['num_heads'],768)
            self.proj_norm = nn.LayerNorm(768)
            self.classifier = nn.Linear(768,2)
            self.dropout = nn.Dropout(.1)

        def encode_genes(self, input_ids, attention_mask, species_ids):
            assert species_ids.shape == (input_ids.shape[0],)
            out = self.ntv3(input_ids=input_ids, attention_mask=attention_mask,
                            species_ids=species_ids, output_hidden_states=True)
            emb = out.hidden_states[-1][:,0,:]
            assert emb.shape[-1] == 768, 'Foundation architecture mismatch'
            return self.embed_dropout(self.layer_norm(emb))

        def contexts(self, gene_embeddings):
            weights = [torch.softmax(head(gene_embeddings),dim=0) for head in self.attention_heads]
            contexts = [(w * gene_embeddings).sum(0) for w in weights]
            return torch.cat(contexts,dim=0), weights

        def classify(self, gene_embeddings):
            context, weights = self.contexts(gene_embeddings)
            z = self.proj_norm(self.projection(context))
            return self.classifier(self.dropout(z.unsqueeze(0))), weights, z

    def encode_bag(model, seqs):
        tok = tokenizer(seqs, padding='max_length', truncation=True,
                        max_length=CFG['max_seq_len'], return_tensors='pt')
        ids = tok['input_ids'].to(device)
        mask = tok.get('attention_mask', torch.ones_like(tok['input_ids'])).to(device)
        embeddings = []
        for start in range(0,len(seqs),CFG['gene_sub_batch']):
            end = min(start+CFG['gene_sub_batch'],len(seqs))
            species = torch.full((end-start,),CFG['species_id'],dtype=torch.long,device=device)
            embeddings.append(model.encode_genes(ids[start:end],mask[start:end],species))
        return torch.cat(embeddings)

    def objective(logits, label, weights):
        y = torch.tensor([label],dtype=torch.long,device=device)
        ce = F.cross_entropy(logits,y,reduction='none')
        alpha = CFG['focal_alpha'] if label else 1-CFG['focal_alpha']
        focal = (alpha * (1-torch.exp(-ce))**CFG['focal_gamma'] * ce).mean()
        diversity = sum(F.cosine_similarity(weights[i].T,weights[j].T,dim=-1).mean()
                        for i in range(len(weights)) for j in range(i+1,len(weights)))
        return focal + CFG['ortho_lambda'] * diversity

    def extract(model, frame):
        model.eval()
        probabilities, features = [], []
        with torch.no_grad():
            for row in frame.itertuples():
                with amp_context():
                    logits, _, z = model.classify(encode_bag(model,row.sequences))
                probabilities.append(torch.softmax(logits,dim=1)[0,1].item())
                features.append(z.float().cpu().numpy())
        X = np.stack(features)
        assert X.shape == (len(frame),768) and np.isfinite(X).all()
        return np.asarray(probabilities), X

    def fit_xgb(Xt, yt, Xv, yv, seed):
        params = {**CFG['xgb'], 'random_state':seed}
        rounds = params.pop('early_stopping_rounds')
        major = int(xgb.__version__.split('.')[0])
        if major >= 2:
            clf = xgb.XGBClassifier(**params, early_stopping_rounds=rounds)
            clf.fit(Xt, yt, eval_set=[(Xv,yv)], verbose=False)
        else:
            assert 'early_stopping_rounds' in inspect.signature(xgb.XGBClassifier.fit).parameters
            clf = xgb.XGBClassifier(**params)
            clf.fit(Xt, yt, eval_set=[(Xv,yv)], early_stopping_rounds=rounds, verbose=False)
        assert clf.classes_.tolist() == [0,1]
        assert clf.best_iteration >= 0
        return clf

    def xgb_probability(clf, X):
        return clf.predict_proba(X, iteration_range=(0,clf.best_iteration+1))[:,1]

    def memory_smoke_test(model, training_frame, directory, seed):
        # Uses only inner train. Repeats selected sequences to stress configured MAX_GENES;
        # pads/truncates to MAX_SEQ_LEN, but cannot cover every data-dependent remote kernel.
        row = training_frame.loc[training_frame.bag_size.idxmax()]
        seqs = (row.sequences * ((CFG['max_genes'] + len(row.sequences)-1)//len(row.sequences)))[:CFG['max_genes']]
        model.train()
        model.zero_grad(set_to_none=True)
        scaler = make_scaler()
        if device.type == 'cuda': torch.cuda.reset_peak_memory_stats(device)
        try:
            with amp_context():
                logits, weights, genome_embedding = model.classify(encode_bag(model,seqs))
                loss = objective(logits,int(row.label),weights)
            assert torch.isfinite(loss), 'Nonfinite smoke loss'
            scaler.scale(loss).backward()
            encoder_grads = [p.grad for p in model.ntv3.parameters() if p.grad is not None]
            assert encoder_grads and any(torch.count_nonzero(g).item() for g in encoder_grads), (
                'No nonzero backbone gradients: abort rather than freeze encoder silently')
            assert all(torch.isfinite(g).all() for g in encoder_grads), 'Nonfinite smoke gradients'
            save_json(directory/'memory_smoke_test.json',dict(status='passed',
                genome_id=str(row.genome_id),genes=len(seqs),max_seq_len=CFG['max_seq_len'],
                amp=AMP_ENABLED,gradient_checkpointing=CFG['gradient_checkpointing'],
                device=str(device),peak_allocated_bytes=int(torch.cuda.max_memory_allocated(device))
                if device.type == 'cuda' else None,optimizer_step=False))
        finally:
            model.zero_grad(set_to_none=True)
        del scaler, logits, weights, loss, genome_embedding, encoder_grads
        gc.collect()
        if device.type == 'cuda': torch.cuda.empty_cache()
        seed_all(seed)  # Smoke dropout/RNG must not change subsequent training order.


In [ ]:
def metrics(y, p):
    y, p = np.asarray(y,dtype=int), np.asarray(p,dtype=float)
    assert len(y) == len(p) and len(y) > 0
    assert set(np.unique(y)) <= {0,1} and np.isfinite(p).all() and ((p >= 0)&(p <= 1)).all()
    tn, fp, fn, tp = map(int,confusion_matrix(y,p >= .5,labels=[0,1]).ravel())
    both = len(np.unique(y)) == 2
    return dict(n=len(y), roc_auc=float(roc_auc_score(y,p)) if both else None,
        pr_auc_ap=float(average_precision_score(y,p)) if both else None,
        TN=tn, FP=fp, FN=fn, TP=tp, VME_count=fn, ME_count=fp,
        resistant_n=tp+fn, susceptible_n=tn+fp,
        VME=fn/(tp+fn) if tp+fn else None, ME=fp/(tn+fp) if tn+fp else None,
        sensitivity=tp/(tp+fn) if tp+fn else None, specificity=tn/(tn+fp) if tn+fp else None)

if CFG['run_training']:
    def validate_oof(frame, expected):
        assert not frame.genome_id.duplicated().any()
        assert frame.genome_id.tolist() == expected.genome_id.tolist()
        assert frame.label.tolist() == expected.label.tolist() and frame.ST.tolist() == expected.ST.tolist()
        for col in ['MIL_probability','XGB_probability']:
            metrics(frame.label, frame[col])
        assert frame.threshold.eq(.5).all()

    def shap_export(clf, Xv, validation, directory, model):
        import shap
        assert model.projection.in_features == 768 * CFG['num_heads']
        assert model.projection.out_features == model.classifier.in_features == Xv.shape[1] == 768
        n = min(len(Xv),CFG['shap_max_validation'])
        assert n > 0
        # SHAP explains the selected trees, not unused post-early-stop trees.
        booster = clf.get_booster()[:clf.best_iteration+1]
        explainer = shap.TreeExplainer(booster)
        values = np.asarray(explainer.shap_values(Xv[:n]))
        assert values.shape == (n,Xv.shape[1]), 'Unexpected binary SHAP output shape'
        names = [f'projected_embedding_{i:03d}' for i in range(Xv.shape[1])]
        pd.DataFrame({'feature':names,'mean_abs':np.abs(values).mean(0),
                      'mean_signed_positive_margin':values.mean(0)}).to_csv(directory/'shap_summary.csv',index=False)
        np.save(directory/'shap_inner_val_values.npy',values)
        validation.iloc[:n][['genome_id','label','ST']].to_csv(directory/'shap_validation_ids.csv',index=False)
        by_class = []
        for label in [0,1]:
            mask = validation.label.iloc[:n].to_numpy() == label
            if mask.any():
                for j,name in enumerate(names):
                    by_class.append(dict(label=label,n=int(mask.sum()),feature=name,
                        mean_abs=float(np.abs(values[mask,j]).mean()),
                        mean_signed_positive_margin=float(values[mask,j].mean())))
        pd.DataFrame(by_class).to_csv(directory/'shap_by_observed_class.csv',index=False)
        save_json(directory/'feature_mapping.json', dict(dimension=Xv.shape[1],features=names,
            meaning='LayerNorm(projection(concat(attention head contexts))); coordinates mix all heads',
            pre_projection_heads=[dict(head=h,start=h*768,end_exclusive=(h+1)*768)
                                  for h in range(CFG['num_heads'])],
            shap_space='binary positive-class raw log-odds; not probabilities or gene mechanisms'))

    verify_saved_execution('def validate_oof(frame, expected):')
    selected_plans = {key:value for key,value in plans.items()
        if (EXECUTION_REQUEST['scheme'] == 'both' or key[0] == EXECUTION_REQUEST['scheme'])
        and (EXECUTION_REQUEST['fold'] is None or key[1] == EXECUTION_REQUEST['fold'])}
    if EXECUTION_REQUEST['mode'] == 'analyze': selected_plans = {}
    for (scheme,fold), indices in selected_plans.items():
        assert implementation_hash(CFG['notebook_path']) == IMPLEMENTATION_HASH, 'Saved code changed during run'
        directory = ROOT / f'{scheme}_fold_{fold}'
        directory.mkdir(exist_ok=True)
        assert not directory.is_symlink(), 'Fold directory must not redirect writes'
        frame = {role:cohort.iloc[idx].reset_index(drop=True) for role,idx in indices.items()}
        split_ids = {role:f.genome_id.tolist() for role,f in frame.items()}
        fold_identity = dict(fingerprint=FINGERPRINT, scheme=scheme, fold=fold, ids=split_ids)
        done_path = directory / 'complete.json'
        if done_path.exists():
            assert CFG['resume'], 'Fold already complete; enable resume or choose new run_id'
            done = json.loads(done_path.read_text())
            assert done['identity'] == fold_identity
            assert {'oof.csv','mil_best.pt','xgb.json','fold_identity.json','params.json','metrics.json'} <= set(done['artifacts'])
            for name, sha in done['artifacts'].items():
                assert Path(name).name == name and file_sha(directory/name) == sha
            old = pd.read_csv(directory/'oof.csv',dtype={'genome_id':str,'ST':str})
            validate_oof(old,frame['outer_test'])
            assert old.scheme.eq(scheme).all() and old.fold.eq(fold).all()
            print('Verified completed fold:', scheme, fold)
            continue
        # Incomplete folders may be retrained from foundation, never used as supervised initialization.
        save_json(directory/'fold_identity.json',fold_identity)
        save_json(directory/'params.json',CFG)
        seed = CFG['seed'] + fold
        seed_all(seed)
        model = NTv3MultiHeadMIL().to(device)
        assert len(model.attention_heads) == CFG['num_heads']
        if CFG['run_memory_smoke_test'] or EXECUTION_REQUEST['mode'] == 'smoke':
            memory_smoke_test(model,frame['inner_train'],directory,seed)
        if EXECUTION_REQUEST['mode'] == 'smoke':
            del model
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            continue  # No optimizer, checkpoint, outer evaluation or completion marker.
        scaler = make_scaler()
        optimizer = torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])
        steps = len(frame['inner_train']) * CFG['max_epochs']
        scheduler = get_cosine_schedule_with_warmup(optimizer,int(steps*CFG['warmup_frac']),steps)
        best, stale, history, best_epoch = -np.inf, 0, [], None
        rng = np.random.default_rng(seed)
        for epoch in range(1,CFG['max_epochs']+1):
            model.train()
            losses = []
            for i in rng.permutation(len(frame['inner_train'])):
                row = frame['inner_train'].iloc[i]
                optimizer.zero_grad(set_to_none=True)
                with amp_context():
                    logits, weights, genome_embedding = model.classify(encode_bag(model,row.sequences))
                    loss = objective(logits,int(row.label),weights)
                assert torch.isfinite(loss), 'Nonfinite training loss'
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                old_scale = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= old_scale: scheduler.step()
                losses.append(loss.item())
                del logits, weights, loss, genome_embedding
                optimizer.zero_grad(set_to_none=True)
            pv, _ = extract(model,frame['inner_val'])
            auc = roc_auc_score(frame['inner_val'].label,pv)
            history.append(dict(epoch=epoch,train_loss=float(np.mean(losses)),inner_val_auc=float(auc)))
            pd.DataFrame(history).to_csv(directory/'training_log.csv',index=False)
            if auc > best:
                best, stale, best_epoch = auc, 0, epoch
                torch.save({'state_dict':model.state_dict(),'identity':fold_identity,
                            'best_epoch':epoch,'inner_val_auc':float(auc)},directory/'mil_best.pt')
            else: stale += 1
            print(scheme,fold,epoch,'inner AUC',round(auc,4))
            if stale >= CFG['patience']: break
        optimizer.zero_grad(set_to_none=True)
        del optimizer, scheduler, scaler
        gc.collect()
        if device.type == 'cuda': torch.cuda.empty_cache()
        checkpoint = torch.load(directory/'mil_best.pt',map_location='cpu',weights_only=True)
        assert checkpoint['identity'] == fold_identity
        model.load_state_dict(checkpoint['state_dict'])
        del checkpoint
        gc.collect()
        for param in model.parameters(): param.requires_grad_(False)
        model.eval()
        _, Xt = extract(model,frame['inner_train'])
        _, Xv = extract(model,frame['inner_val'])
        clf = fit_xgb(Xt,frame['inner_train'].label.to_numpy(),Xv,frame['inner_val'].label.to_numpy(),seed)
        clf.save_model(directory/'xgb.json')
        save_json(directory/'xgb_selection.json',dict(best_iteration=int(clf.best_iteration),
                  evaluation=clf.evals_result(), fit_ids=split_ids['inner_train'], early_stop_ids=split_ids['inner_val']))
        if CFG['run_shap']: shap_export(clf,Xv,frame['inner_val'],directory,model)
        # Outer test is first evaluated only after BOTH models have been selected.
        pm, Xo = extract(model,frame['outer_test'])
        px = xgb_probability(clf,Xo)
        output = frame['outer_test'][['genome_id','label','ST']].copy()
        output['scheme'], output['fold'], output['threshold'] = scheme, fold, .5
        output['MIL_probability'], output['XGB_probability'] = pm, px
        output['MIL_prediction'], output['XGB_prediction'] = (pm >= .5).astype(int), (px >= .5).astype(int)
        validate_oof(output,frame['outer_test'])
        output.to_csv(directory/'oof.csv',index=False)
        save_json(directory/'metrics.json',{m:metrics(output.label,output[f'{m}_probability']) for m in ['MIL','XGB']})
        if CFG['biological_columns']:
            # Explicit binary features only; no mechanism inference, no outer fit.
            bt = frame['inner_train'][CFG['biological_columns']].to_numpy(dtype=float)
            bv = frame['inner_val'][CFG['biological_columns']].to_numpy(dtype=float)
            bo = frame['outer_test'][CFG['biological_columns']].to_numpy(dtype=float)
            bio = fit_xgb(bt,frame['inner_train'].label.to_numpy(),bv,frame['inner_val'].label.to_numpy(),seed)
            bio.save_model(directory/'biological_xgb.json')
            biological = output[['genome_id','label','ST','scheme','fold']].copy()
            biological['probability'] = xgb_probability(bio,bo)
            biological.to_csv(directory/'biological_oof.csv',index=False)
            save_json(directory/'biological_metrics.json',metrics(biological.label,biological.probability))
            save_json(directory/'biological_columns.json',CFG['biological_columns'])
        # Completion marker is written LAST and hashes every fold artifact.
        artifacts = {p.name:file_sha(p) for p in directory.iterdir() if p.is_file() and p.name != 'complete.json'}
        save_json(done_path,dict(identity=fold_identity,artifacts=artifacts))
        del model, clf, Xt, Xv, Xo
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print('Training disabled: model/XGB/checkpoint/SHAP cells did not execute.')

## 4. OOF, errores clínicos y bootstrap pareado por ST

ROC AUC y PR AUC se reportan como **average precision** (no área trapezoidal); matriz fija `[0,1]`. VME = FN / resistentes reales; ME = FP / susceptibles reales, con sus numeradores y denominadores. Métricas por fold y pooled no ajustan el umbral.

Bootstrap de clusters: muestrear ST con reemplazo y mantener todas sus observaciones, incluyendo multiplicidad. Se comparte la muestra entre MIL/XGB y entre random/ST-blocked; se estiman CIs percentiles y diferencias pareadas. Un ST desconocido es un solo cluster. Réplicas de una clase no tienen AUC/AP y se contabilizan como no válidas; no se fabrican p-values. CIs son condicionales a estas OOF y no incorporan variabilidad de reentrenamiento, ni convierten cinco folds en cinco observaciones independientes. Pocos clusters pueden hacer los intervalos poco informativos.

El análisis **lineage-matched** existente compara únicamente genomas outer-test cuyo ST aparece en el outer train del esquema random, para ambos esquemas. Es un análisis **exploratorio post-hoc de OOF**, no una nueva validación ni un criterio para seleccionar modelos; el denominador matched y métricas resistentes se guardan explícitamente.

**Matched-sensitivity ROC (distinto del lineage-matched):** en las OOF outer pooled de la misma cohorte, cada modelo usa el umbral empírico más alto con sensibilidad ≥ `target_sensitivity` (default .95). Los empates no se fraccionan: la sensibilidad alcanzada puede exceder el objetivo y diferir entre modelos; ambos usan el mismo objetivo y los mismos IDs. Se reportan especificidad, FPR, matriz, numeradores/denominadores y diferencia de especificidad MIL−XGB por esquema. Es exclusivamente una comparación exploratoria post-hoc de puntos ROC, **no validación de un umbral para despliegue clínico**. Usa etiquetas outer para describir la curva, no para seleccionar modelos ni cambiar las OOF primarias a .5.


In [ ]:
def empirical_sensitivity_point(y, p, target):
    y, p = np.asarray(y,dtype=int), np.asarray(p,dtype=float)
    assert 0 < target <= 1 and y.shape == p.shape and len(y) > 0
    assert set(np.unique(y)) == {0,1}, 'Matched-sensitivity ROC needs both classes'
    assert np.isfinite(p).all() and ((p >= 0)&(p <= 1)).all()
    # Highest threshold whose empirical TPR reaches target; ties remain indivisible.
    resistant = np.sort(p[y == 1])[::-1]
    needed = int(np.ceil(target * len(resistant)))
    threshold = float(resistant[needed-1])
    predicted = p >= threshold
    tp = int((predicted & (y == 1)).sum()); fn = int(((~predicted) & (y == 1)).sum())
    fp = int((predicted & (y == 0)).sum()); tn = int(((~predicted) & (y == 0)).sum())
    return dict(target_sensitivity=float(target),threshold=threshold,
                sensitivity=tp/(tp+fn),specificity=tn/(tn+fp),FPR=fp/(tn+fp),
                TP=tp,FN=fn,FP=fp,TN=tn,resistant_n=tp+fn,susceptible_n=tn+fp,
                rule='highest empirical threshold reaching target; >=; no tie splitting',
                analysis='exploratory_outer_OOF_matched_sensitivity_ROC',clinical_deployment=False)

def all_folds_verified():
    if EXECUTION_REQUEST['mode'] not in {'train','analyze'}: return False
    missing = [f'{s}_fold_{f}' for s,f in plans if not (ROOT/f'{s}_fold_{f}'/'complete.json').is_file()]
    if missing:
        assert EXECUTION_REQUEST['mode'] != 'analyze', f'Analysis requires all ten completed folds: {missing}'
        print('Pooled analysis deferred; missing folds:', missing)
        return False
    for (scheme,fold), indices in plans.items():
        directory = ROOT/f'{scheme}_fold_{fold}'
        done = json.loads((directory/'complete.json').read_text())
        ids = {role:cohort.iloc[idx].genome_id.tolist() for role,idx in indices.items()}
        assert done['identity'] == dict(fingerprint=FINGERPRINT,scheme=scheme,fold=fold,ids=ids)
        assert {'oof.csv','mil_best.pt','xgb.json','fold_identity.json','params.json','metrics.json'} <= set(done['artifacts'])
        for name,sha in done['artifacts'].items():
            assert Path(name).name == name and file_sha(directory/name) == sha, 'Completed artifact changed'
        old = pd.read_csv(directory/'oof.csv',dtype={'genome_id':str,'ST':str})
        validate_oof(old,cohort.iloc[indices['outer_test']])
        assert old.scheme.eq(scheme).all() and old.fold.eq(fold).all()
    return True

if CFG['run_training'] and all_folds_verified():
    verify_saved_execution('def empirical_sensitivity_point(y, p, target):')
    oof = pd.concat([pd.read_csv(ROOT/f'{s}_fold_{f}'/'oof.csv',dtype={'genome_id':str,'ST':str})
                     for s,f in plans],ignore_index=True)
    for scheme in ['random','st_blocked']:
        current = oof.loc[oof.scheme.eq(scheme)]
        assert len(current) == len(cohort) and not current.genome_id.duplicated().any()
        check = current.merge(cohort[['genome_id','label','ST']],on='genome_id',validate='one_to_one',suffixes=('','_expected'))
        assert set(current.genome_id) == set(cohort.genome_id)
        assert check.label.eq(check.label_expected).all() and check.ST.eq(check.ST_expected).all()
    oof.to_csv(ROOT/'oof_all.csv',index=False)
    metric_rows = []
    for scheme, group in oof.groupby('scheme'):
        for fold, f in list(group.groupby('fold')) + [('pooled',group)]:
            for model in ['MIL','XGB']:
                metric_rows.append(dict(scheme=scheme,fold=fold,model=model,
                                        **metrics(f.label,f[f'{model}_probability'])))
    pd.DataFrame(metric_rows).to_csv(ROOT/'metrics_fold_and_pooled.csv',index=False)

    wide = cohort[['genome_id','label','ST']].copy()
    for scheme in ['random','st_blocked']:
        f = oof.loc[oof.scheme.eq(scheme),['genome_id','MIL_probability','XGB_probability']]
        f = f.rename(columns={f'{m}_probability':f'{scheme}_{m}' for m in ['MIL','XGB']})
        wide = wide.merge(f,on='genome_id',validate='one_to_one')
    score_cols = [f'{s}_{m}' for s in ['random','st_blocked'] for m in ['MIL','XGB']]
    groups = [np.flatnonzero(wide.ST.to_numpy() == st) for st in sorted(wide.ST.unique())]
    rng = np.random.default_rng(CFG['bootstrap_seed'])
    boot_rows = []
    measures = ['roc_auc','pr_auc_ap','sensitivity','specificity','VME','ME']
    comparisons = [('random_MIL','random_XGB'),('st_blocked_MIL','st_blocked_XGB'),
                   ('random_MIL','st_blocked_MIL'),('random_XGB','st_blocked_XGB')]
    for b in range(CFG['bootstrap_B']):
        idx = np.concatenate([groups[i] for i in rng.integers(0,len(groups),size=len(groups))])
        sample = wide.iloc[idx]
        results = {col:metrics(sample.label,sample[col]) for col in score_cols}
        for col in score_cols:
            for measure in measures:
                boot_rows.append(dict(replicate=b,contrast=col,measure=measure,value=results[col][measure]))
        for a,c in comparisons:
            for measure in measures:
                va, vc = results[a][measure], results[c][measure]
                boot_rows.append(dict(replicate=b,contrast=f'{a} minus {c}',measure=measure,
                                      value=va-vc if va is not None and vc is not None else None))
    boot = pd.DataFrame(boot_rows)
    boot.to_csv(ROOT/'bootstrap_replicates.csv',index=False)
    point = {col:metrics(wide.label,wide[col]) for col in score_cols}
    ci_rows = []
    alpha = (1-CFG['ci_level'])/2
    for (contrast,measure), f in boot.groupby(['contrast','measure']):
        valid = f.value.dropna()
        if ' minus ' in contrast:
            a,c = contrast.split(' minus ')
            estimate = point[a][measure]-point[c][measure]
        else: estimate = point[contrast][measure]
        ci_rows.append(dict(contrast=contrast,measure=measure,estimate=estimate,
            lower=float(valid.quantile(alpha)) if len(valid) else None,
            upper=float(valid.quantile(1-alpha)) if len(valid) else None,
            valid_B=len(valid),invalid_B=CFG['bootstrap_B']-len(valid),clusters=len(groups),
            ci_level=CFG['ci_level'],seed=CFG['bootstrap_seed']))
    pd.DataFrame(ci_rows).to_csv(ROOT/'paired_cluster_ST_CI.csv',index=False)

    # This is ROC operating-point matching, NOT lineage matching or deployable tuning.
    sensitivity_rows, sensitivity_comparisons = [], []
    for scheme, group in oof.groupby('scheme'):
        assert set(group.genome_id) == set(cohort.genome_id)
        points = {}
        for model in ['MIL','XGB']:
            point = empirical_sensitivity_point(group.label,group[f'{model}_probability'],CFG['target_sensitivity'])
            points[model] = point
            sensitivity_rows.append(dict(scheme=scheme,model=model,n=len(group),**point))
        sensitivity_comparisons.append(dict(scheme=scheme,
            target_sensitivity=CFG['target_sensitivity'],
            MIL_sensitivity=points['MIL']['sensitivity'],XGB_sensitivity=points['XGB']['sensitivity'],
            specificity_difference_MIL_minus_XGB=points['MIL']['specificity']-points['XGB']['specificity'],
            same_outer_genome_ids=True,clinical_deployment=False,
            analysis='exploratory_outer_OOF_matched_sensitivity_ROC'))
    pd.DataFrame(sensitivity_rows).to_csv(ROOT/'exploratory_matched_sensitivity_ROC_points.csv',index=False)
    pd.DataFrame(sensitivity_comparisons).to_csv(ROOT/'exploratory_matched_sensitivity_ROC_comparison.csv',index=False)

    matched_ids = []
    for (scheme,fold), indices in plans.items():
        if scheme != 'random': continue
        train_st = set(cohort.iloc[np.concatenate([indices['inner_train'],indices['inner_val']])].ST)
        test = cohort.iloc[indices['outer_test']]
        matched_ids.extend(test.loc[test.ST.isin(train_st),'genome_id'])
    matched = oof.loc[oof.genome_id.isin(matched_ids)].copy()
    matched.to_csv(ROOT/'exploratory_outer_matched_oof.csv',index=False)
    matched_rows = []
    for scheme, f in matched.groupby('scheme'):
        for model in ['MIL','XGB']:
            matched_rows.append(dict(analysis='exploratory_outer_random_lineage_matched',scheme=scheme,
                                     model=model,**metrics(f.label,f[f'{model}_probability'])))
    pd.DataFrame(matched_rows).to_csv(ROOT/'exploratory_outer_matched_metrics.csv',index=False)
    save_json(ROOT/'exploratory_matched_definition.json',dict(n=len(set(matched_ids)),
        total_n=len(cohort),definition='random outer-test ST present in random outer-train; same IDs for both schemes'))
    print(pd.DataFrame(metric_rows).to_string(index=False))
else:
    print('No OOF or metric estimates created: preflight is not an experiment.')

## 5. Asociación aadA2 opcional y límites de interpretación

`aada2_column` debe apuntar a una columna **booleana explícita del manifiesto** curada externamente. No se infiere desde cadenas Drug Class, nombres de genes o secuencias. Las tablas cuentan presencia/ausencia × fenotipo por ST y pooled; no se interpreta aadA2 como mecanismo de resistencia al carbapenémico ni se generan pruebas/p-values. Este análisis descriptivo usa la cohorte completa e incluye etiquetas, pero jamás altera bolsas, cohortes ni entrenamiento.

SHAP opcional explica las 768 coordenadas proyectadas de XGB sobre IDs **inner validation** guardados. Se guardan magnitud media absoluta y signo del margen positivo, pooled y por clase observada. Coordenadas proyectadas mezclan las cuatro cabezas; el mapa de cabezas solo corresponde a las 3072 entradas pre-proyección, no a genes. SHAP no demuestra causalidad biológica. IG **no está disponible**: no se presenta una atribución sobre IDs de tokens discretos ni un baseline biológico sin validar la ruta de embeddings de NTv3.

Coocurrencia adicional requiere `carbapenemase_column`, otra columna booleana explícita del manifiesto validada igual que aadA2. Se guardan las cuatro combinaciones aadA2 × carbapenemasa pooled y por ST, con numerador, denominador total y denominadores condicionales. No se sustituye presencia de carbapenemasa por fenotipo; los ratios con denominador cero son nulos. La asociación aadA2 × fenotipo anterior permanece como tabla descriptiva separada.


In [ ]:
if CFG['aada2_column']:
    column = CFG['aada2_column']
    def association_counts(frame, st_name):
        out = []
        for present in [0,1]:
            for label in [0,1]:
                out.append(dict(ST=st_name,aadA2_present=present,label=label,
                    n=int(((frame[column] == present)&(frame.label == label)).sum())))
        return out
    rows = association_counts(cohort,'__POOLED__')
    for st, group in cohort.groupby('ST'): rows.extend(association_counts(group,st))
    pd.DataFrame(rows).to_csv(ROOT/'aadA2_descriptive_counts_by_ST.csv',index=False)
    save_json(ROOT/'aadA2_analysis.json',dict(column=column,analysis='descriptive association only',
        cohort_n=len(cohort),mechanism_inference=False,p_values_computed=False))
else:
    print('aadA2 association disabled: no explicit manifest column configured.')
print('IG unavailable by design; no placeholder attributions generated.')

if CFG['aada2_column'] and CFG['carbapenemase_column']:
    def cooccurrence_counts(frame, st_name, aada2_column, carbapenemase_column):
        # Presence × presence, not phenotype × presence. All four cells, including zeros.
        rows = []
        for a in [0,1]:
            a_den = int(frame[aada2_column].eq(a).sum())
            for c in [0,1]:
                c_den = int(frame[carbapenemase_column].eq(c).sum())
                count = int((frame[aada2_column].eq(a) & frame[carbapenemase_column].eq(c)).sum())
                rows.append(dict(ST=st_name,aadA2_present=a,carbapenemase_present=c,
                    numerator=count,cohort_denominator=len(frame),
                    aadA2_stratum_denominator=a_den,carbapenemase_stratum_denominator=c_den,
                    joint_proportion=count/len(frame) if len(frame) else None,
                    carbapenemase_status_given_aadA2_status=count/a_den if a_den else None,
                    aadA2_status_given_carbapenemase_status=count/c_den if c_den else None))
        return rows
    a, c = CFG['aada2_column'], CFG['carbapenemase_column']
    rows = cooccurrence_counts(cohort,'__POOLED__',a,c)
    for st, group in cohort.groupby('ST'): rows.extend(cooccurrence_counts(group,st,a,c))
    pd.DataFrame(rows).to_csv(ROOT/'aadA2_carbapenemase_cooccurrence_pooled_ST.csv',index=False)
    save_json(ROOT/'aadA2_carbapenemase_definition.json',dict(aada2_column=a,carbapenemase_column=c,
        phenotype_used=False,analysis='descriptive explicit-boolean cooccurrence',
        mechanism_inference=False,p_values_computed=False))
else:
    print('aadA2/carbapenemase cooccurrence disabled: requires both explicit boolean columns.')


## CLI reproducible (sin depender del historial IPython)

El runner ejecuta todas las celdas de código exactas en orden. Inyecta `RUN_OVERRIDES` antes de CFG, valida claves, guarda overrides y su SHA256 e incluye ambos y el hash del runner en la identidad inicial. Instalación deshabilitada por defecto en CLI; Drive se monta por separado por la sesión padre. Preflight/smoke escriben bajo `validated_results/preflight` y `validated_results/smoke`; train/analyze comparten `validated_results/lineage_validation_20261013`.

La cohorte automática raw sigue siendo el default. `cohort_selection` es opcional y solo cruza nombres originales exactos (`nombre_archivo_original` por defecto), sin inferir equivalencias de IDs. Un filtro histórico columna/valor debe ser explícito. Duplicados, ausentes, conflictos de fenotipo o bolsas no elegibles abortan y conservan auditoría; no se reclama igualdad histórica sin este cruce. Ambos esquemas usan siempre la misma cohorte seleccionada.

## Checklist de ejecución y artefactos

1. Ejecutar con las rutas reales por defecto: la preparación genera automáticamente el manifiesto explícito desde carpetas S/R originales y ST_table accession/ST. Resolver solo errores reales de datos o mapeos ambiguos que el preflight reporte; no crear CSV de entrada ficticios.
2. Los defaults de rutas ya están extraídos del flujo existente. Usar un run_id nuevo si hay una ejecución incompatible previa. Preflight crea hashes SHA256 de archivos, identidad de configuración/implementación, entorno, cohorte, exclusiones, selección de bolsas y asignaciones de folds. Si no hay cinco folds válidos se detiene con error, no inventa particiones.
3. Para entrenar, habilitar consentimiento de código remoto revisado y `run_training`; el SHA se resuelve y bloquea automáticamente, y usar un **run_id nuevo** respecto del preflight si cambió cualquier configuración de identidad. Ejecutar desde el inicio. No hay claves ni secretos en el cuaderno; autenticación Hugging Face, si hiciera falta, debe hacerse externamente.
4. Por fold: IDs, parámetros, checkpoint MIL, log de epochs/inner AUC, modelo XGB, early stopping, OOF con label/fold/ST/umbral, métricas y marcador final con hashes. Tokenizador y versiones se guardan a nivel de run. Reanudar verifica identidad, versiones, hashes e IDs; folds incompletos se recomputan desde foundation. No hay reanudación a mitad de época.
5. Final: OOF completas, métricas pooled/per-fold, réplicas bootstrap y CI pareadas, sensibilidad matched exploratoria; baseline y SHAP opcionales. Un resultado nulo indica métrica no definida, no cero. **No se generan resultados ficticios en preflight.**

Limitaciones: dependencia del ST disponible y posible confusión residual dentro/fuera de linajes; el único grupo desconocido puede ser grande. No es validación temporal/externa; sin ajuste por centro, ni corrección causal. El archivo de selección contiene secuencias genómicas: proteger el Drive y los outputs según las políticas de datos. SHA y seeds ayudan a reproducibilidad, no garantizan igualdad bit a bit GPU. Verificar compatibilidad del commit NTv3 con el entorno antes de un experimento costoso. Las dependencias opcionales son pins convencionales, no una afirmación de compatibilidad NTv3 ya probada.

**Identidad de implementación y Colab:** para train o `resume=True`, guardar todos los cambios y reiniciar **Run all**. Se intenta localizar la copia guardada mediante metadata Colab y candidatos existentes cuyo código coincida con el editor. Si no se identifica inequívocamente, la **única excepción de ruta manual** es `notebook_path` al `.ipynb` guardado real en Drive; no se adivina su ubicación. Se hashea en ejecución el contenido exacto de todas las celdas de código, no outputs/markdown. No hay string de versión manual. Antes de descargar NTv3, entrenar/reanudar y agregar OOF, se comprueba que la copia guardada no cambió y que el historial IPython ejecutó el código guardado en orden; redefiniciones no guardadas de componentes del protocolo abortan. El runner `run_lineage_validation.py` habilita scripts CLI verificando el hash del notebook, el hash del runner y el prefijo ordenado de celdas exactas; no modifica ni transforma el código ejecutado. Preflight sin `notebook_path` tiene identidad de código nula y no puede reutilizarse como run de entrenamiento. Cambios de código/configuración requieren otra carpeta; los runs anteriores con fingerprint fijo son incompatibles. Para reanudar, conservar la configuración y los RUN_OVERRIDES de identidad; `resume` y `run_training` son controles excluidos de identidad. El selector de esquema/fold se registra por invocación y no cambia el fingerprint. Pooled/bootstrap solo se ejecuta tras verificar los diez folds completos de ambos esquemas.


El fingerprint de datos contiene hashes de **todos los raw RGI descubiertos y todas las ST_table realmente leídas**, además del inventario, la elección de fuentes y el CSV histórico de selección cuando se solicita; no hashea CSV generados como si fueran fuentes independientes. Se exportan inventario raw, auditorías de archivo/fila, mapa ST sin match y secuencias con trazabilidad original. Si los ST disponibles no permiten cinco folds válidos, el cuaderno aborta antes de entrenar, sin fabricar grupos. Preflight no genera resultados ni consume foundation; sí puede instalar dependencias convencionales y montar Drive.
